Instances

We can create instances by calling a class like it were a function: ```i = ClassName(...)```. Then parameters given in the call will be passed to the ```__init__``` function. In the ```__init__``` method you can create the instance specific attributes. If ```__init__``` is missing, we can create an instance without giving any parameters. As a consequence, the instance has no attributes. Later you can (re)bind attributes with the assignment instance.attribute = new value.

If that attribute did not exist before, it will be added to the instance with the assigned value. In Python we really can add or delete attributes to/from an existing instance. This is possible because the attribute names and the corresponding values are actually stored in a dictionary. This dictionary is also an attribute of the instance and is called dict. Another standard attribute in addition to dict is called ```__class__```. This attribute stores the class of the instance. That is, the type of the object

## Instances Can Grow New Attributes — And Here's Where They Actually Live

This section reveals the mechanism behind everything you just learned about `self.a = ...` — and it connects **directly** to something you already know deeply: dictionaries!

---

### "We Really Can Add or Delete Attributes" — Instances Are Flexible

Unlike some languages where an object's shape is fixed forever, Python instances can **grow new attributes on the fly**, even ones the class never mentioned:

```python
class MyClass:
    def __init__(self, param1):
        self.b = param1

i = MyClass(5)
print(i.b)        # → 5

i.z = 100         # ← 'z' was NEVER defined anywhere in the class!
print(i.z)        # → 100    ✓ just... works
```

No error, no special declaration needed. You can even **delete** one:

```python
del i.z
print(i.z)        # ✗ AttributeError — 'z' is gone
```

---

### Why This Works — Instances Store Attributes in a DICTIONARY

Here's the reveal: an instance's attributes aren't stored in some mysterious, fixed structure. They're kept in an ordinary **dictionary**, accessible via `__dict__`:

```python
print(i.__dict__)
# → {'b': 5, 'z': 100}
```

**That dictionary IS the instance's attribute storage.** Every time you write `i.something`, Python is really doing a **dictionary lookup** under the hood:

```
i.b       ≈       i.__dict__['b']
i.z = 100  ≈       i.__dict__['z'] = 100
```

You've been using `.dot` notation this whole course thinking of it as something special — but it's the **same dictionary mechanism** you already mastered, just with a friendlier `.` syntax layered on top!

---

### Verifying It — Watch the Dict Grow in Real Time

```python
class MyClass:
    def __init__(self, param1):
        self.b = param1

i = MyClass(5)
print(i.__dict__)      # → {'b': 5}

i.z = 100
print(i.__dict__)       # → {'b': 5, 'z': 100}    ← 'z' appeared in the dict!

del i.z
print(i.__dict__)        # → {'b': 5}               ← 'z' removed from the dict!
```

Adding an attribute = adding a key to this dict. Deleting an attribute = deleting a key from this dict. It's genuinely that simple — `self.b = param1` inside `__init__` was **always** just `self.__dict__['b'] = param1`, dressed up in nicer syntax.

---

### This Also Explains the "Class Attribute Shadowing" Trap From Last Time!

Remember the `self.a = 99` trap — where writing to `self.a` created a **private copy** instead of modifying the shared class attribute? Now you can see **exactly** why, using `__dict__`:

```python
class MyClass:
    a = 1     # class attribute — lives in MyClass's OWN dict, not the instance's

i = MyClass()
print(i.__dict__)        # → {}     ← empty! 'a' is NOT here — it's on the CLASS

i.a = 99                  # creates a NEW key in i's OWN dict
print(i.__dict__)         # → {'a': 99}   ← now it IS here, shadowing the class's 'a'

print(MyClass.a)          # → 1      ← the class's dict is untouched
```

Two **separate dictionaries** — `i.__dict__` and `MyClass.__dict__` — and `i.a = 99` only ever touches the **instance's** one. This is the mechanical reason behind the entire trap you learned about earlier.

---

### `__class__` — Another Automatic Attribute

Every instance also automatically carries a reference back to **the class it was built from**:

```python
i = MyClass(5)
print(i.__class__)          # → <class '__main__.MyClass'>
print(i.__class__ == MyClass)  # → True
```

This is essentially what `type()` uses internally:

```python
print(type(i))              # → <class '__main__.MyClass'>    (same info)
print(type(i) == i.__class__)   # → True
```

Remember `isinstance(obj, Type)` from your instances lesson? Under the hood, it's basically checking `obj.__class__` against `Type`.

---

### The Full Mental Model, Connected

```
i = MyClass(5)

i                     →  an OBJECT
i.__dict__             →  a DICTIONARY holding i's own attributes: {'b': 5}
i.__class__             →  a reference to MyClass — "what blueprint made me"

i.b                     →  looks in i.__dict__ first  → found → 5
i.a (if 'a' not on i)    →  not in i.__dict__ → falls back to MyClass.__dict__ → found → 1
```

Everything you've learned about dictionaries — keys, values, adding, deleting, `in` checks — applies **directly** to how objects store their attributes. There was never a separate, hidden "attribute mechanism" — it was dictionaries all along, wearing a dot-notation costume.

---

### The One-Sentence Summary

> An instance's attributes are literally stored in an ordinary dictionary called `__dict__`, which is why you can freely add or delete attributes at runtime — `instance.x = value` is really `instance.__dict__['x'] = value` in disguise. Every instance also carries a `__class__` attribute pointing back to the class (type) it was built from — the same information `type()` and `isinstance()` rely on. 🎯